# 01 · Baseline: standard (Euler) residual transformer, 20M params, 50M tokens

1. Build the model and count parameters.
2. Search for the largest batch that fits, then **fix** the batch size to the largest power of two at or below it. Runs 01 and 02 both use this batch size.
3. Train for 50M tokens and save loss, tokens/s and peak memory to `results/baseline.json`.

In [ ]:
# --- Colab setup: GPU runtime (Runtime > Change runtime type > T4 GPU) ---
REPO_URL = "https://github.com/gaurkhare/gaurav-eagv5-s13.git"
import os, sys, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era_a13"          # data + results persist across notebooks
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    os.chdir("/content/repo")
else:
    WORK = os.path.abspath("..")                      # running locally from notebooks/
    os.chdir(WORK)
sys.path.insert(0, os.getcwd())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA_DIR, RESULTS = f"{WORK}/data", f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no nvidia GPU"

In [ ]:
import torch
from revllm import LLM, ModelConfig, TrainConfig, train, find_max_batch
mcfg = ModelConfig(residual="euler")
m = LLM(mcfg)
print(m)
print(f"params: {m.num_params():,}  (embedding, tied: {m.embed.weight.numel():,})")
del m

## Find the batch size the baseline can run

In [ ]:
bmax, trials = find_max_batch(mcfg, start=8)
B_FIXED = 1 << (bmax.bit_length() - 1)             # largest power of two <= max
print(f"baseline max batch = {bmax}; fixed batch for runs 01/02 = {B_FIXED} ({B_FIXED*mcfg.seq_len:,} tokens/step)")
json.dump({"baseline_max_batch": bmax, "fixed_batch": B_FIXED, "trials": trials}, open(f"{RESULTS}/batch.json", "w"), indent=1)

## Train

In [ ]:
cfg = TrainConfig(name="baseline", data_dir=DATA_DIR, out_dir=RESULTS, batch_size=B_FIXED,
                  total_tokens=50_000_000, lr=1e-3, model=mcfg)
res, model = train(cfg)

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; e = res["evals"]
plt.figure(figsize=(8, 4))
plt.plot([r["tokens"]/1e6 for r in h], [r["loss"] for r in h], label="train", alpha=.6)
plt.plot([r["tokens"]/1e6 for r in e], [r["val_loss"] for r in e], "o-", label="val")
plt.xlabel("tokens (M)"); plt.ylabel("loss"); plt.ylim(top=6); plt.legend(); plt.grid(alpha=.3); plt.title("baseline")
plt.show()
{k: res[k] for k in ["final_train_loss", "final_val_loss", "tokens_per_s", "peak_mem_allocated_gb", "peak_mem_reserved_gb", "wall_time_min", "gpu"]}

In [ ]:
# sample generation (sanity check)
from transformers import AutoTokenizer
from revllm.data import TOKENIZER
tok = AutoTokenizer.from_pretrained(TOKENIZER)
@torch.no_grad()
def generate(model, prompt, n=60, temp=0.8):
    model.eval(); dev = next(model.parameters()).device
    x = torch.tensor([tok(prompt)["input_ids"]], device=dev)
    for _ in range(n):
        logits = model(x[:, -mcfg.seq_len:])[:, -1] / temp
        x = torch.cat([x, torch.multinomial(logits.float().softmax(-1), 1)], 1)
    return tok.decode(x[0])
print(generate(model, "The water cycle is"))